# 数据准备、审计与真实 batch 检查（成员 A）

用途：在真实 SYSU-MM01 上运行 A 的数据接口——有数据就直接用，没有就自动下载；生成 manifest 与审计；取真实 batch 检查接口。后续 EDA（E1–E5）也在本 notebook 继续。

- 代码本体在 `src/data.py`（B、E 会 import 它），本 notebook 只调用，不复制代码。
- 运行前已安装 `numpy pillow torch torchvision kagglehub`。没有数据时第 1 步会自动下载（约 2 GB）。
- 从上到下依次运行。第一次生成 manifest 要逐张检查约 4.6 万张图，需要几分钟。
- **提交 Git 前请清空输出**（菜单 Edit → Clear All Outputs）：输出里有数据集图片，仓库是公开的，不应把数据集图像传上去。

In [ ]:
import os, sys
from pathlib import Path

# 找到项目根目录（含 src/data.py 的目录），并切换过去，保证相对路径一致
ROOT = Path.cwd()
while not (ROOT / "src" / "data.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy, PIL, torch, torchvision
from src import data as D
print("project root:", ROOT)
print("python", sys.version.split()[0], "| torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| numpy", numpy.__version__, "| Pillow", PIL.__version__)

## 1. 准备数据：有就直接用，没有就下载；生成 manifest 与审计

查找顺序：环境变量 `SYSU_MM01_ROOT` → `data/SYSU-MM01` → 上次自动下载的位置；都没有时用 kagglehub 下载。
已有 manifest 时直接复用；要重新扫描把 `REBUILD` 改成 True。手动下载的数据可填写 `DOWNLOADED_ON`，会写进审计报告。

In [ ]:
REBUILD = False
SOURCE = "https://www.kaggle.com/datasets/coconutjean/sysumm01"
DOWNLOADED_ON = None      # 手动下载时改成 "2026-10-24" 这样的日期；自动下载会自动记录

prep = D.prepare_data(download=True, rebuild=REBUILD, source=SOURCE, downloaded_on=DOWNLOADED_ON)
print("dataset:", prep["root"])
print("audit:", prep["audit_status"])

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown((prep["manifest_dir"] / "data_audit.md").read_text(encoding="utf-8")))

## 2. 取真实训练 batch，检查接口契约

输出到 `artifacts/results/data_check/`。`num_workers=0` 在 Windows 上更稳，另外会用 2 个 worker 对照一次，确认结果一致。

In [ ]:
sys.path.insert(0, str(ROOT / "scripts"))
from check_data import run_checks
CHECK_DIR = ROOT / "artifacts" / "results" / "data_check"
passed = run_checks(prep["root"], prep["manifest"], "M0", out_dir=CHECK_DIR, num_workers=0)
print("ALL PASSED" if passed else "SOME CHECKS FAILED")

In [ ]:
display(Markdown((CHECK_DIR / "check_report.md").read_text(encoding="utf-8")))
# 每行一个身份：左 K 张为可见光（第一阶段已转灰度，标 g），右 K 张为同一身份的 IR。请人工确认同一行是同一个人。
display(Image(filename=str(CHECK_DIR / "batch_grid.png"), width=700))

## 3. 交互查看（可选）

直接使用 B 将调用的接口 `get_train_loader`，看一个 batch 的字段，并对比第一阶段（灰度）与第二阶段（彩色）同一步抽到的图片。

In [ ]:
from collections import Counter

rows = D.load_manifest(prep["manifest"])
print("images per split/modality:", dict(sorted(Counter((r["split"], r["modality"]) for r in rows).items())))

loader, sampler, dataset, label_map = D.get_train_loader("M0", num_workers=0, pin_memory=False)
P, K = sampler.P, sampler.K
print(f"P={P} K={K} batch={sampler.batch_size} steps/epoch={sampler.batches_per_epoch} classes={len(label_map)}")

sampler.set_epoch(0)
sampler.set_gray_probability(1.0)      # 第一阶段
stage1 = next(iter(loader))
sampler.set_epoch(0)
sampler.set_gray_probability(0.0)      # M0 第二阶段
stage2 = next(iter(loader))
D.check_pmt_layout(stage1, P, K); D.check_pmt_layout(stage2, P, K)
print("same images in both stages:", stage1["paths"] == stage2["paths"])
for k, v in stage1.items():
    print(f"  {k:12s}", tuple(v.shape) if hasattr(v, "shape") else f"list[{len(v)}]")

In [ ]:
from PIL import Image as PILImage

def to_pil(t):
    mean = torch.tensor(D.IMAGENET_MEAN).view(3, 1, 1); std = torch.tensor(D.IMAGENET_STD).view(3, 1, 1)
    return PILImage.fromarray((t * std + mean).clamp(0, 1).mul(255).byte().permute(1, 2, 0).numpy())

# 第一个身份：上排第一阶段（RGB 灰度 + IR），下排第二阶段（RGB 彩色 + IR）
n = P * K
idx = list(range(K)) + list(range(n, n + K))
canvas = PILImage.new("RGB", (len(idx) * 132, 2 * 260), "white")
for r, b in enumerate((stage1, stage2)):
    for c, i in enumerate(idx):
        canvas.paste(to_pil(b["images"][i]), (c * 132, r * 260))
print("identity", stage1["identities"][0].item(), "| cameras", stage1["cameras"][idx].tolist())
display(canvas)

## 4. 下一步：EDA（内部目标 10/26–10/28）

按设计文档 §3.4 补充 E1–E5，每张图写“观察—影响—处理决策”：

- E1 每摄像头/模态/划分的图片数与身份数（`data_audit.json` 已有数量）
- E2 每身份样本数、身份×摄像头覆盖热图
- E3 尺寸、宽高比、亮度、对比度分布（manifest 已含 `mean_intensity`、`std_intensity`、`channel_spread`）
- E4 同身份 RGB/IR、相似外观不同身份、遮挡样例
- E5 重复/损坏统计与身份集合交集

主要分析 train/val，不用测试集做设计决策。